# Sprint 3 — Fine-Tuning LoRA/QLoRA : Adapter un LLM à HomeButler

> **Environnement recommandé** : Google Colab (GPU T4 gratuit)  
> **Durée estimée** : 20-30 min (dont ~15 min d'entraînement)  
> **Objectif** : Fine-tuner Mistral 7B sur notre dataset HomeButler pour éliminer les hallucinations sans RAG

## Ce que vous allez apprendre

1. La différence entre Full Fine-Tuning, LoRA et QLoRA
2. Pourquoi QLoRA permet de fine-tuner un 7B sur un GPU T4 (16 Go)
3. Comment préparer un dataset d'instruction au format Mistral
4. Comment suivre un entraînement avec MLFlow
5. Comment évaluer avant/après avec ROUGE-L
6. Comment exporter en GGUF pour Ollama


---
## Section 1 — Théorie : Full Fine-Tuning vs LoRA vs QLoRA

### Le problème du Full Fine-Tuning

Mistral 7B a **7 milliards de paramètres**. En FP32, ça fait ~28 Go rien que pour les poids.
Pour le fine-tuning, on a besoin aussi des gradients (~28 Go) et des états de l'optimiseur (~56 Go).
**Total : ~112 Go de VRAM** — soit 4 GPU A100 pour un seul modèle.

| Approche | Params modifiés | VRAM min | Durée (7B) | Cas d'usage |
|---|---|---|---|---|
| **Full fine-tuning** | 100% (7B params) | 80 Go+ | 10h+ | Réentraînement complet |
| **LoRA** | ~0.1% (matrices Q, V) | 16 Go | 1-2h | Adaptation domaine |
| **QLoRA** | ~0.1% en 4-bit | 8-10 Go | 1-2h | GPU accessible (T4) |

### L'idée de LoRA (Low-Rank Adaptation)

Au lieu de modifier tous les poids W (7B params), LoRA **ajoute deux petites matrices** A et B :

```
W_new = W_original + α × (B × A)
       ↑ gelé       ↑ entraîné
```

- `W_original` : **gelé** (on ne touche pas aux 7B paramètres)
- `B` : matrice (dim × r) avec r = rang (ex: 8)
- `A` : matrice (r × dim) initialisée à zéro
- `α` : facteur de mise à l'échelle (ex: 16)

Pour r=8 sur les matrices Q et V d'un modèle 7B → **~4 millions de paramètres entraînés** au lieu de 7 milliards.

### QLoRA : LoRA + quantization 4-bit

QLoRA charge les poids gelés en **4-bit NF4** (au lieu de 16-bit) :
- 7B en FP16 = ~14 Go
- 7B en 4-bit NF4 = **~4 Go**
- + adaptateurs LoRA + gradients = **~8-10 Go total**
- Tient dans un **GPU T4 (16 Go)** ✅


---
## Section 2 — Setup Google Colab

> ⚠️ Vérifiez que vous avez bien sélectionné un **GPU T4** dans Colab : Runtime → Change runtime type → T4 GPU


In [ ]:
# Vérification GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'GPU détecté : {result.stdout.strip()}')
else:
    print('⚠️  Aucun GPU détecté ! Allez dans Runtime → Change runtime type → T4 GPU')

In [ ]:
# Installation des dépendances (prend ~3 minutes sur Colab)
!pip install -q \
    transformers==4.45.0 \
    peft==0.13.0 \
    trl==0.11.0 \
    bitsandbytes==0.43.3 \
    datasets==3.0.0 \
    mlflow==2.16.0 \
    accelerate==0.34.0 \
    rouge-score==0.1.2

print('Installation terminée !')

In [ ]:
# Optionnel : monter Google Drive pour sauvegarder les checkpoints
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_DIR = '/content/drive/MyDrive/homebutler-lora'

OUTPUT_DIR = '/content/homebutler-lora'
print(f'Checkpoints sauvegardés dans : {OUTPUT_DIR}')

---
## Section 3 — Chargement et préparation du dataset

On utilise notre dataset HomeButler augmenté (généré par `scripts/augment_qa_dataset.py`).
Format attendu : chaque paire est convertie au **format instruction Mistral** :
```
<s>[INST] {instruction}\n{question} [/INST] {réponse} </s>
```


In [ ]:
import json
import random
from collections import Counter

# Télécharger le dataset depuis GitHub (adapter l'URL si fork privé)
# En local : charger directement depuis le repo
DATASET_URL = 'https://raw.githubusercontent.com/YoLoADR/pre-training-rag/main/data/qa_dataset/augmented_concierge_qa.jsonl'

import urllib.request
try:
    with urllib.request.urlopen(DATASET_URL) as response:
        lines = response.read().decode('utf-8').strip().split('\n')
    pairs = [json.loads(line) for line in lines if line.strip()]
    print(f'Dataset chargé depuis GitHub : {len(pairs)} paires')
except Exception as e:
    print(f'GitHub non accessible ({e}). Tentative locale...')
    try:
        with open('../data/qa_dataset/augmented_concierge_qa.jsonl', encoding='utf-8') as f:
            pairs = [json.loads(line) for line in f if line.strip()]
        print(f'Dataset local chargé : {len(pairs)} paires')
    except FileNotFoundError:
        print('Fichier non trouvé. Chargement du dataset original...')
        ORIG_URL = 'https://raw.githubusercontent.com/YoLoADR/pre-training-rag/main/data/qa_dataset/concierge_qa.jsonl'
        with urllib.request.urlopen(ORIG_URL) as response:
            lines = response.read().decode('utf-8').strip().split('\n')
        pairs = [json.loads(line) for line in lines if line.strip()]
        print(f'Dataset original chargé : {len(pairs)} paires')

# Distribution par catégorie
categories = Counter(p.get('category', 'autre') for p in pairs)
print('\nDistribution par catégorie :')
for cat, n in sorted(categories.items(), key=lambda x: -x[1]):
    print(f'  {cat:<20} {n:>4} ({100*n/len(pairs):.1f}%)')

In [ ]:
# Formatage au format instruction Mistral
def format_mistral_instruction(pair: dict) -> str:
    """Convertit une paire (instruction, input, output) au format Mistral."""
    instruction = pair.get('instruction', 'Tu es HomeButler, conciergerie domestique bienveillante.')
    question = pair.get('input', '')
    answer = pair.get('output', '')
    return f'<s>[INST] {instruction}\n{question} [/INST] {answer} </s>'

formatted = [format_mistral_instruction(p) for p in pairs]

# Vérification
print('Exemple formaté :')
print(formatted[0][:300])
print('...')
print(f'\nLongueur moyenne : {sum(len(f) for f in formatted) / len(formatted):.0f} chars')
print(f'Longueur max     : {max(len(f) for f in formatted)} chars')

In [ ]:
from datasets import Dataset

# Mélange reproductible
random.seed(42)
random.shuffle(pairs)

# Split train/test : 90/10
split_idx = int(len(pairs) * 0.9)
train_pairs = pairs[:split_idx]
test_pairs = pairs[split_idx:]

# Créer les datasets HuggingFace
train_dataset = Dataset.from_list([
    {'text': format_mistral_instruction(p)} for p in train_pairs
])
eval_dataset = Dataset.from_list([
    {'text': format_mistral_instruction(p)} for p in test_pairs
])

print(f'Train : {len(train_dataset)} exemples')
print(f'Test  : {len(eval_dataset)} exemples')

---
## Section 4 — Chargement du modèle de base en QLoRA 4-bit

On charge Mistral 7B avec **BitsAndBytes 4-bit NF4** pour réduire l'empreinte mémoire de ~14 Go à ~4 Go.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.2'

# Configuration QLoRA 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,               # quantization 4-bit
    bnb_4bit_compute_dtype=torch.float16,  # calculs en FP16
    bnb_4bit_use_double_quant=True,  # double quantization (économise ~0.4 bits/param)
    bnb_4bit_quant_type='nf4',       # NormalFloat4 (meilleur pour modèles LLM)
)

print(f'Chargement de {MODEL_NAME}...')
print('(peut prendre 3-5 minutes selon la connexion)')

# Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Mistral n'a pas de pad token natif
tokenizer.padding_side = 'right'           # pour éviter les warnings SFTTrainer

# Chargement du modèle en 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',      # répartit automatiquement sur GPU(s) disponibles
    trust_remote_code=True,
)
model.config.use_cache = False  # incompatible avec gradient checkpointing

# Rapport mémoire
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'\nGPU : {torch.cuda.get_device_name(0)}')
    print(f'Mémoire utilisée : {mem_used:.1f} Go / {mem_total:.1f} Go')
    print(f'Modèle 4-bit chargé ✅ (vs ~14 Go en FP16)')

# Compter les paramètres
total_params = sum(p.numel() for p in model.parameters())
print(f'Paramètres totaux : {total_params / 1e9:.2f} B (gelés en 4-bit)')

---
## Section 5 — Configuration LoRA

On définit les adaptateurs LoRA : **quelles matrices** modifier, avec quel **rang** `r`.

### Impact du rang `r` sur le nombre de paramètres entraînés

Pour une couche d'attention Mistral (dim=4096) :

| r (rang) | Params LoRA par couche | Total (32 couches × 2 matrices) | % du modèle |
|---|---|---|---|
| 4 | 2 × (4096×4 + 4×4096) = 65,536 | ~4.2 M | 0.06% |
| **8** | 2 × (4096×8 + 8×4096) = 131,072 | **~8.4 M** | **0.12%** |
| 16 | 2 × (4096×16 + 16×4096) = 262,144 | ~16.8 M | 0.24% |
| 64 | 2 × (4096×64 + 64×4096) = 1,048,576 | ~67.1 M | 0.96% |

**r=8** est le bon compromis pour une adaptation domaine légère comme HomeButler.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Préparer le modèle pour l'entraînement en 4-bit (active gradient checkpointing)
model = prepare_model_for_kbit_training(model)

# Configuration LoRA
lora_config = LoraConfig(
    r=8,                                    # rang des matrices adaptatives
    lora_alpha=16,                          # facteur de mise à l'échelle (souvent 2×r)
    target_modules=['q_proj', 'v_proj'],    # matrices Q et V de l'attention multi-tête
    lora_dropout=0.05,                      # régularisation
    bias='none',                            # ne pas entraîner les biais
    task_type='CAUSAL_LM',                  # génération de texte causal
)

# Appliquer LoRA au modèle
model = get_peft_model(model, lora_config)

# Rapport des paramètres entraînables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Paramètres entraînables : {trainable:,} ({100 * trainable / total:.3f}% du total)')
print(f'Paramètres gelés        : {total - trainable:,}')
print(f'Total                   : {total:,}')
model.print_trainable_parameters()

---
## Section 6 — Entraînement avec SFTTrainer + MLFlow

`SFTTrainer` (Supervised Fine-Tuning Trainer) de la librairie `trl` est optimisé pour l'instruction fine-tuning.
MLFlow permet de tracker les métriques d'entraînement (loss, learning rate) et de comparer des runs.


In [ ]:
import mlflow
from trl import SFTTrainer
from transformers import TrainingArguments

# Configuration MLFlow (tracking local dans /content/mlruns)
mlflow.set_tracking_uri('file:///content/mlruns')
mlflow.set_experiment('homebutler-finetuning')

# Hyperparamètres d'entraînement
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,      # batch de 4 exemples
    gradient_accumulation_steps=4,      # accumule 4 batches → batch effectif = 16
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=True,                           # calculs en FP16 pour la vitesse
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,                  # garde seulement les 2 derniers checkpoints
    load_best_model_at_end=True,
    report_to='none',                    # on gère le tracking via MLFlow manuellement
    optim='paged_adamw_32bit',          # optimiseur paginé (économise VRAM)
    lr_scheduler_type='cosine',
)

# Démarrage du run MLFlow
with mlflow.start_run(run_name='mistral-homebutler-qlora') as run:
    # Log des hyperparamètres
    mlflow.log_params({
        'model': MODEL_NAME,
        'lora_r': 8,
        'lora_alpha': 16,
        'target_modules': 'q_proj,v_proj',
        'epochs': 3,
        'batch_size': 4,
        'gradient_accumulation': 4,
        'learning_rate': 2e-4,
        'train_size': len(train_dataset),
        'eval_size': len(eval_dataset),
        'quantization': '4-bit NF4',
    })

    # Entraînement
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=lora_config,
        dataset_text_field='text',
        max_seq_length=512,
        tokenizer=tokenizer,
        args=training_args,
    )

    print('Démarrage de l\'entraînement...')
    print(f'Durée estimée : ~15-20 min sur T4 pour 3 epochs sur {len(train_dataset)} exemples')

    train_result = trainer.train()

    # Log des métriques finales
    final_loss = train_result.training_loss
    mlflow.log_metrics({
        'final_train_loss': final_loss,
        'train_runtime_seconds': train_result.metrics.get('train_runtime', 0),
        'train_samples_per_second': train_result.metrics.get('train_samples_per_second', 0),
    })

    # Sauvegarde des adaptateurs LoRA
    trainer.save_model(OUTPUT_DIR)
    print(f'\nAdaptateurs LoRA sauvegardés dans {OUTPUT_DIR}')
    print(f'Loss finale : {final_loss:.4f}')
    print(f'MLFlow Run ID : {run.info.run_id}')

In [ ]:
# Visualiser les métriques MLFlow
# Pour lancer l'UI : !mlflow ui --host 0.0.0.0 --port 5000 &
# Puis utiliser ngrok ou le port forwarding Colab

# Récupérer les métriques via l'API MLFlow
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name('homebutler-finetuning')
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['start_time DESC'],
    max_results=5
)

print('=== Runs MLFlow ===\n')
for r in runs:
    print(f'Run : {r.info.run_name}')
    print(f'  Status  : {r.info.status}')
    for k, v in r.data.params.items():
        print(f'  Param   : {k} = {v}')
    for k, v in r.data.metrics.items():
        print(f'  Metric  : {k} = {v:.4f}')
    print()

---
## Section 7 — Évaluation : avant/après + ROUGE-L

On compare le modèle **base** (sans fine-tuning) et le modèle **fine-tuné** sur les mêmes 5 questions.
Puis on calcule ROUGE-L sur l'ensemble de test pour une évaluation quantitative.


In [ ]:
from transformers import pipeline as hf_pipeline
from peft import PeftModel

# Questions de test (représentatives des 4 catégories)
QUESTIONS_EVAL = [
    {
        'input': 'Mon lave-linge fait un bruit bizarre lors de l\'essorage, que faire ?',
        'category': 'équipements',
    },
    {
        'input': 'Quelle est la température idéale pour la chaudière la nuit ?',
        'category': 'équipements',
    },
    {
        'input': 'Mon propriétaire peut-il entrer dans mon appartement sans prévenir ?',
        'category': 'droits',
    },
    {
        'input': 'Ma consommation électrique a doublé ce mois-ci, pourquoi ?',
        'category': 'énergie',
    },
    {
        'input': 'Où trouver du pain artisanal bio livré près de chez moi ?',
        'category': 'marketplace',
    },
]

INSTRUCTION = 'Tu es HomeButler, conciergerie domestique bienveillante. Réponds à la question suivante sur le logement.'


def generate_response(model_or_pipeline, question: str, max_new_tokens: int = 250) -> str:
    """Génère une réponse avec le modèle donné."""
    prompt = f'<s>[INST] {INSTRUCTION}\n{question} [/INST]'
    if hasattr(model_or_pipeline, '__call__'):
        # pipeline HuggingFace
        result = model_or_pipeline(prompt, max_new_tokens=max_new_tokens, do_sample=False)[0]
        generated = result['generated_text'].split('[/INST]')[-1].strip()
    else:
        # modèle + tokenizer direct
        inputs = tokenizer(prompt, return_tensors='pt').to(model_or_pipeline.device)
        with torch.no_grad():
            outputs = model_or_pipeline.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id
            )
        generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated.strip()


print('=== Évaluation qualitative — Avant / Après fine-tuning ===\n')
for item in QUESTIONS_EVAL:
    question = item['input']
    category = item['category']

    # Réponse du modèle fine-tuné (chargé en mémoire depuis l'entraînement)
    response_ft = generate_response(model, question)

    print(f'Catégorie : [{category}]')
    print(f'Question : {question}')
    print(f'FINE-TUNÉ : {response_ft[:300]}')
    print('-' * 70)

In [ ]:
# Évaluation quantitative : ROUGE-L sur l'ensemble de test
from rouge_score import rouge_scorer as rouge_module

scorer = rouge_module.RougeScorer(['rougeL'], use_stemmer=True)

print(f'Calcul ROUGE-L sur {len(test_pairs)} exemples de test...')

rouge_scores = []
for i, pair in enumerate(test_pairs[:30]):  # limiter à 30 pour la vitesse
    question = pair.get('input', '')
    reference = pair.get('output', '')

    # Générer la réponse
    prediction = generate_response(model, question, max_new_tokens=200)

    # Calculer ROUGE-L
    score = scorer.score(reference, prediction)['rougeL'].fmeasure
    rouge_scores.append(score)

    if (i + 1) % 10 == 0:
        print(f'  [{i+1}/30] ROUGE-L moyen jusqu\'ici : {sum(rouge_scores)/len(rouge_scores):.3f}')

avg_rouge = sum(rouge_scores) / len(rouge_scores)
print(f'\n=== Résultats ROUGE-L ===')
print(f'Score moyen : {avg_rouge:.3f}')
print(f'Score min   : {min(rouge_scores):.3f}')
print(f'Score max   : {max(rouge_scores):.3f}')
print()
print('Interprétation :')
print('  ROUGE-L > 0.5 : réponses très proches de la référence (excellente mémorisation)')
print('  ROUGE-L 0.3-0.5 : réponses correctes avec paraphrase (adapté à notre cas)')
print('  ROUGE-L < 0.3 : réponses trop différentes de la référence (sous-entraînement)')

# Log dans MLFlow
try:
    with mlflow.start_run(run_id=run.info.run_id):
        mlflow.log_metric('avg_rouge_l', avg_rouge)
except:
    pass  # run déjà terminé, pas critique

---
## Section 8 — Distillation de modèles (concept)

La distillation est une technique pour **comprimer les connaissances d'un grand modèle (teacher) dans un petit modèle (student)**.

### Principe

```
Teacher (Claude claude-sonnet-4-6, 100B+)  →  génère des "soft labels" (distributions de probabilité)
Student (Phi-3 mini 3.8B)                   →  apprend à reproduire ces distributions
```

La **distillation loss** combine :
- **Cross-entropy** : reproduire la réponse du teacher
- **KL divergence** : reproduire la distribution de probabilités du teacher (pas juste le token le plus probable)

```python
# Pseudo-code distillation
loss = alpha * cross_entropy(student_logits, ground_truth)
     + (1 - alpha) * kl_divergence(student_logits / T, teacher_logits / T)
# T = température (T > 1 "adoucit" les distributions pour transfert de connaissance)
```

### Pourquoi la distillation ?

| Scénario | Résultat |
|---|---|
| Fine-tuner Mistral 7B (notre TP) | 7B params, ~4 Go GGUF, bonne qualité |
| Distiller vers Phi-3 mini (3.8B) | 3.8B params, ~2.3 Go GGUF, qualité préservée |
| Distiller vers TinyLlama (1.1B) | 1.1B params, ~0.7 Go GGUF, qualité moindre |

> **Ce TP ne couvre pas la distillation** (nécessite un GPU A100 et l'accès à l'API Claude).
> Ce concept est mentionné car il est au programme RAFT J2 et représente la prochaine étape
> naturelle après le fine-tuning LoRA.


In [ ]:
# Illustration conceptuelle de la distillation (non exécutable sans API)

DISTILLATION_PSEUDOCODE = '''
# === Génération des labels par le Teacher (Claude via API) ===
import anthropic
client = anthropic.Anthropic(api_key="...")  # teacher = Claude claude-sonnet-4-6

def get_teacher_response(question: str) -> str:
    """Le teacher génère une réponse de haute qualité = label synthétique."""
    msg = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": f"[HomeButler concierge] {question}"}]
    )
    return msg.content[0].text

# Pour 150 questions → 150 réponses Claude = dataset de distillation
distillation_dataset = [
    {"input": q, "output": get_teacher_response(q)}
    for q in homebutler_questions
]

# === Entraînement du Student (Phi-3 mini 3.8B) ===
# Même configuration LoRA mais sur un modèle plus petit
student = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",  # student = 3.8B
    quantization_config=bnb_config,
    device_map="auto"
)
# Fine-tuner le student sur les réponses du teacher
# → Le student apprend le "style" HomeButler sans avoir besoin du RAG
'''

print('Distillation : Teacher → Student')
print('=' * 50)
print(DISTILLATION_PSEUDOCODE)
print()
print('Résultat attendu :')
print('  - Student (3.8B) répond comme HomeButler')
print('  - Taille : ~2.3 Go GGUF (vs ~4 Go pour Mistral 7B fine-tuné)')
print('  - Vitesse : ~2x plus rapide à l\'inférence')

---
## Section 9 — Export GGUF + Déploiement Ollama

Après fine-tuning, on exporte le modèle en **GGUF 4-bit** pour le déployer avec Ollama.

GGUF = format de quantization développé par llama.cpp — permet de faire tourner des LLMs sur CPU.

### Pipeline d'export
```
LoRA adaptateurs  →  merge avec base  →  Mistral-7B merged (FP16, ~14 Go)
                  →  llama.cpp convert  →  homebutler-q4.gguf (Q4_K_M, ~4.1 Go)
                  →  Ollama Modelfile   →  ollama run homebutler
```


In [ ]:
# Étape 1 : Merger les adaptateurs LoRA dans le modèle de base
from peft import PeftModel

print('Merger LoRA dans le modèle de base...')
print('(Décommentez et exécutez si vous avez terminé l\'entraînement)')

MERGE_PSEUDOCODE = '''
# Charger le modèle de base EN FP16 (pas 4-bit) pour le merge
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cpu',  # CPU pour éviter les problèmes de mémoire GPU lors du merge
)

# Charger les adaptateurs LoRA et merger
model_merged = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model_merged = model_merged.merge_and_unload()  # fusionne LoRA dans les poids

# Sauvegarder le modèle mergé en FP16
MERGED_DIR = '/content/homebutler-merged'
model_merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f'Modèle mergé sauvegardé dans {MERGED_DIR}')
print(f'Taille : ~14 Go (FP16)')
'''
print(MERGE_PSEUDOCODE)

In [ ]:
# Étape 2 : Conversion GGUF avec llama.cpp
# À exécuter sur votre machine locale (ou Colab avec espace disque suffisant)

GGUF_COMMANDS = '''
# === Sur votre machine locale ===

# 1. Cloner llama.cpp
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp && make -j$(nproc)
pip install -r requirements.txt

# 2. Convertir en GGUF FP16
python convert_hf_to_gguf.py ./homebutler-merged \\
    --outtype f16 \\
    --outfile homebutler-f16.gguf

# 3. Quantizer en Q4_K_M (meilleur rapport qualité/taille)
./llama-quantize homebutler-f16.gguf homebutler-q4km.gguf Q4_K_M
# Résultat : ~4.1 Go (vs ~14 Go en FP16)
ls -lh homebutler-q4km.gguf

# === Comparaison des formats de quantization ===
# Q8_0   : ~7.7 Go  | qualité quasi-identique FP16 | trop lourd
# Q6_K   : ~6.1 Go  | excellente qualité           | recommandé si RAM suffisante
# Q4_K_M : ~4.1 Go  | très bonne qualité           | ← notre choix (bon compromis)
# Q4_K_S : ~3.9 Go  | bonne qualité
# Q3_K_M : ~3.3 Go  | qualité acceptable           | pour machines limitées
# Q2_K   : ~2.8 Go  | qualité dégradée             | déconseillé
'''
print(GGUF_COMMANDS)

In [ ]:
# Étape 3 : Déploiement avec Ollama

OLLAMA_COMMANDS = '''
# === Déploiement Ollama (sur votre machine locale) ===

# 1. Créer un Modelfile (définit le comportement du modèle)
cat > Modelfile << 'EOF'
FROM ./homebutler-q4km.gguf

SYSTEM """Tu es HomeButler, la conciergerie domestique intelligente.
Tu connais parfaitement le logement de tes utilisateurs :
- Chaudière Viessmann Vitodens 100-W
- Appartement à Boulogne-Billancourt
- Producteurs locaux bio dans le quartier
Réponds toujours de manière bienveillante, pratique et concise."""

PARAMETER temperature 0.1
PARAMETER num_ctx 2048
PARAMETER stop "[INST]"
PARAMETER stop "</s>"
EOF

# 2. Créer le modèle Ollama
ollama create homebutler -f Modelfile

# 3. Tester
ollama run homebutler "Quelle est la marque de ma chaudière ?"

# 4. Basculer l'API HomeButler vers le modèle local
# Dans votre .env :
# LLM_PROVIDER=ollama
# OLLAMA_MODEL=homebutler
# OLLAMA_BASE_URL=http://localhost:11434

# → Relancer l'API : uvicorn api.main:app --reload
# → Toute l'application utilise maintenant votre modèle fine-tuné local !
'''
print(OLLAMA_COMMANDS)

print('\n=== Récapitulatif du pipeline complet ===')
print('''
150 paires QA HomeButler
        ↓  augment_qa_dataset.py
~430 paires augmentées
        ↓  Section 3 — formatage Mistral
Dataset d\'instruction formaté
        ↓  Section 4-6 — QLoRA + SFTTrainer
Adaptateurs LoRA (8.4M params, ~15 Mo)
        ↓  Section 9 — merge + llama.cpp
homebutler-q4km.gguf (~4.1 Go)
        ↓  Ollama Modelfile
ollama run homebutler  ✅
        ↓  .env LLM_PROVIDER=ollama
API HomeButler 100% locale et fine-tunée ✅
''')

---
## Récapitulatif — Ce que nous avons accompli

| Étape | Concept clé | Résultat |
|---|---|---|
| Section 1 | Full FT vs LoRA vs QLoRA | Comprendre les trade-offs |
| Section 2 | Setup Colab T4 | GPU prêt, dépendances installées |
| Section 3 | Format instruction Mistral | ~430 exemples formatés |
| Section 4 | QLoRA 4-bit NF4 | Modèle 7B en ~8 Go VRAM |
| Section 5 | LoRA r=8, q_proj/v_proj | ~8.4M params entraînables (0.12%) |
| Section 6 | SFTTrainer + MLFlow | Entraînement tracé |
| Section 7 | ROUGE-L avant/après | Évaluation quantitative |
| Section 8 | Distillation | Concept Teacher→Student |
| Section 9 | GGUF + Ollama | Déploiement local |

### Prochaines étapes

1. **Améliorer le dataset** : augmenter à 1000+ paires avec des conversations multi-tours
2. **Essayer des variantes LoRA** : tester `r=16` et `target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]`
3. **Combiner RAG + fine-tuning** : le modèle fine-tuné connaît le style HomeButler, le RAG apporte les données fraîches
4. **Évaluation LLM-as-judge** : utiliser Claude pour évaluer la qualité des réponses (pas seulement ROUGE)
